# Vectorize

In [11]:
import lancedb
import pandas as pd

db = lancedb.connect(uri = "../knowledge_base")
db

LanceDBConnection(uri='c:\\Users\\edvin\\.vscode\\ELLA_fab4\\explorations\\..\\knowledge_base')

In [2]:
db.uri

'c:\\Users\\edvin\\.vscode\\ELLA_fab4\\explorations\\vector_database'

In [4]:
import json
with open("../data/clean_data/brottsbalken_clean.json", encoding="utf-8") as file:
    data = json.load(file)

df = pd.DataFrame(data)
df

,paragraph,text,law_references,chapter_number,title
0,1 §,Brott är en gärning som är beskriven i denna b...,[_Lag (1994:458)_.],1 kap.,Om brott och brottspåföljder
1,2 §,"En gärning skall, om inte annat är särskilt fö...",[_Lag (1994:458)_.],1 kap.,Om brott och brottspåföljder
2,3 §,Med påföljd för brott förstås i denna balk str...,[_Lag (2026:253)_.],1 kap.,Om brott och brottspåföljder
3,4 §,Om användningen av straffen gäller vad i bestä...,[_Lag (1988:942)_.],1 kap.,Om brott och brottspåföljder
4,5 §,Fängelse är att anse som ett svårare straff än...,[_Lag (2026:253)_.],1 kap.,Om brott och brottspåföljder
...,...,...,...,...,...
551,16 §,En begäran om omprövning skall vara skriftlig ...,[_Lag (2005:967)_.],38 kap.,Rättegångsbestämmelser m.m.
552,17 §,Kriminalvården prövar om skrivelsen med begära...,[_Lag (2005:967)_.],38 kap.,Rättegångsbestämmelser m.m.
553,18 §,Beslut som avses i 14 § överklagas till den fö...,[_Lag (2009:776)_.],38 kap.,Rättegångsbestämmelser m.m.
554,19 §,Mål om uppskjuten villkorlig frigivning enligt...,[_Lag (2021:249)_.],38 kap.,Rättegångsbestämmelser m.m.


In [5]:
def build_embed_text(row):
    parts = [
        f"{row['chapter_number']}  {row['title']}",
        f"{row['paragraph']} ",
        row['text'],
        f"{row['law_references']} "
    ]


    return " | ".join(p for p in parts if p)

df["embed_text"] = df.apply(build_embed_text, axis=1)

df

,paragraph,text,law_references,chapter_number,title,embed_text
0,1 §,Brott är en gärning som är beskriven i denna b...,[_Lag (1994:458)_.],1 kap.,Om brott och brottspåföljder,1 kap. Om brott och brottspåföljder | 1 § | ...
1,2 §,"En gärning skall, om inte annat är särskilt fö...",[_Lag (1994:458)_.],1 kap.,Om brott och brottspåföljder,1 kap. Om brott och brottspåföljder | 2 § | ...
2,3 §,Med påföljd för brott förstås i denna balk str...,[_Lag (2026:253)_.],1 kap.,Om brott och brottspåföljder,1 kap. Om brott och brottspåföljder | 3 § | ...
3,4 §,Om användningen av straffen gäller vad i bestä...,[_Lag (1988:942)_.],1 kap.,Om brott och brottspåföljder,1 kap. Om brott och brottspåföljder | 4 § | ...
4,5 §,Fängelse är att anse som ett svårare straff än...,[_Lag (2026:253)_.],1 kap.,Om brott och brottspåföljder,1 kap. Om brott och brottspåföljder | 5 § | ...
...,...,...,...,...,...,...
551,16 §,En begäran om omprövning skall vara skriftlig ...,[_Lag (2005:967)_.],38 kap.,Rättegångsbestämmelser m.m.,38 kap. Rättegångsbestämmelser m.m. | 16 § |...
552,17 §,Kriminalvården prövar om skrivelsen med begära...,[_Lag (2005:967)_.],38 kap.,Rättegångsbestämmelser m.m.,38 kap. Rättegångsbestämmelser m.m. | 17 § |...
553,18 §,Beslut som avses i 14 § överklagas till den fö...,[_Lag (2009:776)_.],38 kap.,Rättegångsbestämmelser m.m.,38 kap. Rättegångsbestämmelser m.m. | 18 § |...
554,19 §,Mål om uppskjuten villkorlig frigivning enligt...,[_Lag (2021:249)_.],38 kap.,Rättegångsbestämmelser m.m.,38 kap. Rättegångsbestämmelser m.m. | 19 § |...


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 556 entries, 0 to 555
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   paragraph       556 non-null    object
 1   text            556 non-null    object
 2   law_references  556 non-null    object
 3   chapter_number  556 non-null    object
 4   title           556 non-null    object
 5   embed_text      556 non-null    object
dtypes: object(6)
memory usage: 26.2+ KB


In [7]:
df["embed_text"][0]

"1 kap.  Om brott och brottspåföljder | 1 §  | Brott är en gärning som är beskriven i denna balk eller i annan lag eller författning och för vilken straff som sägs nedan är föreskrivet. | ['_Lag (1994:458)_.'] "

In [9]:
from lancedb.embeddings import get_registry
from lancedb.pydantic import LanceModel, Vector
from utils.constants import EMBEDDING_MODEL

embedding_model = get_registry().get("cohere").create(name=EMBEDDING_MODEL)

class ParagraphModel(LanceModel):
    embed_text: str = embedding_model.SourceField()
    vector: Vector(embedding_model.ndims()) = embedding_model.VectorField()


    chapter_number: str
    title: str
    law_references: list[str]
    paragraph: str
    text: str  

In [12]:

db = lancedb.connect("../knowledge_base/lancedb")

table = db.create_table("brottsbalken", schema=ParagraphModel, mode="overwrite")


table.add(df[[
    "embed_text",
    "chapter_number", "title", "law_references", "paragraph", "text"
]].to_dict(orient="records"))

print(f"{table.count_rows()} paragrafer indexerade")

556 paragrafer indexerade


In [17]:
table.search("vad är straffet för mord") \
    .limit(5) \
    .to_pandas()[["chapter_number", "title", "paragraph", "text", "law_references", "_distance"]]

,chapter_number,title,paragraph,text,law_references,_distance
0,3 kap.,Om brott mot liv och hälsa,1 §,"Den som berövar annan livet, döms för mord til...",[_Lag (2019:805)_.],0.777316
1,1 kap.,Om brott och brottspåföljder,4 §,Om användningen av straffen gäller vad i bestä...,[_Lag (1988:942)_.],0.825274
2,23 kap.,"Om försök, förberedelse, stämpling och medverk...",5 §,_/Träder i kraft I:2026-07-01/_\nDet får dömas...,[_Lag (2026:94)_.],0.834377
3,23 kap.,"Om försök, förberedelse, stämpling och medverk...",2 §,"_/Upphör att gälla U:2026-07-01/_\nDen som, me...",[_Lag (2016:508)_.],0.845088
4,29 kap.,Om straffmätning och påföljdseftergift,2 a §,Som synnerligen försvårande omständigheter vid...,[_Lag (2023:257)_.],0.849465


In [21]:
df_vectors = table.to_pandas()
df_vectors

,embed_text,vector,chapter_number,title,law_references,paragraph,text
0,1 kap. Om brott och brottspåföljder | 1 § | ...,"[-0.009750366, -0.06970215, -0.12524414, -0.13...",1 kap.,Om brott och brottspåföljder,[_Lag (1994:458)_.],1 §,Brott är en gärning som är beskriven i denna b...
1,1 kap. Om brott och brottspåföljder | 2 § | ...,"[0.026184082, -0.052490234, -0.08959961, -0.15...",1 kap.,Om brott och brottspåföljder,[_Lag (1994:458)_.],2 §,"En gärning skall, om inte annat är särskilt fö..."
2,1 kap. Om brott och brottspåföljder | 3 § | ...,"[-0.03338623, -3.8444996e-05, -0.08581543, -0....",1 kap.,Om brott och brottspåföljder,[_Lag (2026:253)_.],3 §,Med påföljd för brott förstås i denna balk str...
3,1 kap. Om brott och brottspåföljder | 4 § | ...,"[-0.02758789, -0.04714966, -0.06573486, -0.113...",1 kap.,Om brott och brottspåföljder,[_Lag (1988:942)_.],4 §,Om användningen av straffen gäller vad i bestä...
4,1 kap. Om brott och brottspåföljder | 5 § | ...,"[-0.032928467, -0.022064209, -0.1071167, -0.09...",1 kap.,Om brott och brottspåföljder,[_Lag (2026:253)_.],5 §,Fängelse är att anse som ett svårare straff än...
...,...,...,...,...,...,...,...
551,38 kap. Rättegångsbestämmelser m.m. | 16 § |...,"[0.012054443, -0.054840088, -0.118652344, -0.1...",38 kap.,Rättegångsbestämmelser m.m.,[_Lag (2005:967)_.],16 §,En begäran om omprövning skall vara skriftlig ...
552,38 kap. Rättegångsbestämmelser m.m. | 17 § |...,"[0.049713135, -0.046447754, -0.04763794, -0.15...",38 kap.,Rättegångsbestämmelser m.m.,[_Lag (2005:967)_.],17 §,Kriminalvården prövar om skrivelsen med begära...
553,38 kap. Rättegångsbestämmelser m.m. | 18 § |...,"[0.0769043, 0.0068588257, -0.051361084, -0.117...",38 kap.,Rättegångsbestämmelser m.m.,[_Lag (2009:776)_.],18 §,Beslut som avses i 14 § överklagas till den fö...
554,38 kap. Rättegångsbestämmelser m.m. | 19 § |...,"[0.018661499, 0.060638428, -0.13464355, -0.075...",38 kap.,Rättegångsbestämmelser m.m.,[_Lag (2021:249)_.],19 §,Mål om uppskjuten villkorlig frigivning enligt...


In [ ]:
df_vectors.to_json(
    "../data/clean_data/brottsbalken_vectors.json",
    orient="records",
    indent=4,
    force_ascii=False
)